<a href="https://colab.research.google.com/github/parshav42/50_ML_models/blob/main/sugarcnemodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import numpy as np
import pandas as pp
import torchvision
from torchvision import datasets
from torch.utils.data import DataLoader

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.device_count())

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!unzip -b /content/drive/MyDrive/sugrcanenewfileimagecolab.zip  -d /content/dataset/

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT


In [ ]:
from torchvision import transforms

manuly = transforms.Compose([

    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])

In [ ]:
# auto_transform = weights.transforms()
# auto_transform

In [ ]:
import os
import random
import shutil

# Ensure reproducibility by seeding once
random.seed(42)

l = ['healthy','mosaic','redrot','rust','yellow']
output_root_folder = "/content/dataset" # This will be the parent for the 'images' folder

# Create the main 'images/train' and 'images/val' directories once
output_train_base_dir = os.path.join(output_root_folder, "images", "train")
output_val_base_dir = os.path.join(output_root_folder, "images", "val")
os.makedirs(output_train_base_dir, exist_ok=True)
os.makedirs(output_val_base_dir, exist_ok=True)

# Variables to accumulate total counts for summary
total_train_images_processed = 0
total_val_images_processed = 0

# Define a helper function for copying files to avoid repetition
def copy_class_files(image_list, source_folder, destination_folder):
    for image_name in image_list:
        src_path = os.path.join(source_folder, image_name)
        dest_path = os.path.join(destination_folder, image_name)
        shutil.copy(src_path, dest_path)

# Process each class separately
for class_name in l:
    source_class_folder = os.path.join("/content/dataset/train", class_name)

    # Check if the source folder exists
    if not os.path.exists(source_class_folder):
        print(f"Warning: Source folder {source_class_folder} does not exist. Skipping class '{class_name}'.")
        continue

    # Get all images for the current class
    images_in_class = [
        f for f in os.listdir(source_class_folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    if not images_in_class:
        print(f"Warning: No images found in {source_class_folder}. Skipping class '{class_name}'.")
        continue

    # Shuffle and split images for the current class
    random.shuffle(images_in_class)
    split_idx = int(len(images_in_class) * 0.8)
    train_images_for_class = images_in_class[:split_idx]
    val_images_for_class = images_in_class[split_idx:]

    # Create destination subdirectories for the current class within train/val
    dest_train_class_dir = os.path.join(output_train_base_dir, class_name)
    dest_val_class_dir = os.path.join(output_val_base_dir, class_name)
    os.makedirs(dest_train_class_dir, exist_ok=True)
    os.makedirs(dest_val_class_dir, exist_ok=True)

    # Copy training images for the current class
    copy_class_files(train_images_for_class, source_class_folder, dest_train_class_dir)
    total_train_images_processed += len(train_images_for_class)

    # Copy validation images for the current class
    copy_class_files(val_images_for_class, source_class_folder, dest_val_class_dir)
    total_val_images_processed += len(val_images_for_class)

    print(f"Processed class '{class_name}': Train ({len(train_images_for_class)} images), Validation ({len(val_images_for_class)} images)")

print("\n--- Data Splitting and Copying Summary ---")
print("Total Train Images Copied:", total_train_images_processed)
print("Total Validation Images Copied:", total_val_images_processed)


In [ ]:
!rm rf /content/dataset/images

In [ ]:
from pathlib import Path
train = Path('/content/dataset/train')
test = Path('/content/dataset/test')
val = Path('/content/dataset/val')

In [ ]:
!rm -rf /content/dataset/train/.ipynb_checkpoints

In [ ]:
import shutil
import os
from pathlib import Path


train = datasets.ImageFolder(
    train,
    transform = manuly
)

test = datasets.ImageFolder(
    test,
    transform= manuly
)
test = datasets.ImageFolder(
    val,
    transform= manuly
)

In [ ]:
train_dalaloa = DataLoader(
    train,
    batch_size=32,
    shuffle=True,
    drop_last=True,
    pin_memory=True
)

test_dalaloa = DataLoader(
    test,
    batch_size=32,
    shuffle=False
)
val_dalaloa = DataLoader(
    val,
    batch_size=32,
    shuffle=False
)

In [ ]:
#model
model = torchvision.models.efficientnet_b0(weights=weights).to(device)

In [ ]:
!pip install -q torchinfo
from torchinfo import summary
summary(model,input_size=(32,3,224,224))

In [ ]:
t = train.classes

In [ ]:
from torch.nn import parameter
for params in model.features.parameters():
  params.requires_grad = False



In [ ]:
torch.cuda.manual_seed(42)
out = len(t)
model.classifier= torch.nn.Sequential(
    torch.nn.Dropout(0.5),
    torch.nn.Linear(in_features=1280,out_features=out,bias=True)
).to(device)

In [ ]:
summary(model,input_size=(32,3,224,224))

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
epochs=20

# Ensure model is on the correct device if there was a discrepancy
model.to(device)

for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_features, batch_labels in train_dalaloa:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # calculate loss
    loss = loss_fn(outputs, batch_labels)

    # back pass
    optim.zero_grad()
    loss.backward()

    # update grads
    optim.step()

    total_epoch_loss = total_epoch_loss + loss.item()

  avg_loss = total_epoch_loss/len(train_dalaloa)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

In [ ]:
model.eval()

In [ ]:
from sklearn.metrics import confusion_matrix
import torch
import numpy as np # Import numpy for array concatenation
import torchvision.datasets as datasets
from torch.utils.data import DataLoader

total = 0
correct = 0

# Lists to collect all true and predicted labels for the entire dataset
all_true_labels = []
all_predicted_labels = []

# Re-initializing val_dalaloa here to ensure it's a valid DataLoader
# This addresses the TypeError that occurs because `val` was a Path object in the original DataLoader creation.
# The necessary variables (`val`, `manuly`, `datasets`, `DataLoader`) are available from previous cells.
if not isinstance(val, datasets.ImageFolder):
    val_dataset_fixed = datasets.ImageFolder(
        val,
        transform=manuly
    )
    val_dalaloa = DataLoader(
        val_dataset_fixed,
        batch_size=32,
        shuffle=False
    )

with torch.no_grad():

  for batch_features, batch_labels in test_dalaloa: # Iterating over test_dalaloa as per current cell content

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

    # Collect true and predicted labels for overall confusion matrix
    # Append the entire numpy array for each batch, instead of extending with its scalar elements.
    all_true_labels.append(batch_labels.cpu().numpy())
    all_predicted_labels.append(predicted.cpu().numpy())

# Calculate and print the confusion matrix after processing all batches
print("Confusion Matrix:")
# Concatenate the lists of 1D arrays into single 1D numpy arrays
print(confusion_matrix(np.concatenate(all_true_labels), np.concatenate(all_predicted_labels)))

print("\nAccuracy:")
print(correct/total)

In [ ]:
torch.save(model,'sugarcaneacc79.pth')